<a href="https://colab.research.google.com/github/juanfmitzig/RAGFisica1/blob/main/RAG_Fisica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Upgrade v8 — Precisión numérica + DCL + Gráficas
**Asistente RAG de Física — Mecánica · UNS**

Este notebook contiene **solo las celdas nuevas / de reemplazo** para agregar al notebook v7.
No toca la indexación ni los embeddings: todo lo que ya funciona queda igual.

| Sección | Qué agrega | Dónde va en el v7 |
|---|---|---|
| **A** | Verificación numérica con Python (ejecutor + pipeline de 2 pasadas) | Reemplaza las celdas **11 · System prompt** y **12 · `ask_physics()`** |
| **B** | Diagrama de Cuerpo Libre: el LLM describe (JSON) y matplotlib dibuja | Celdas nuevas, después de la sección A |
| **C** | Gráficas $x(t)$, $v(t)$, $a(t)$ por tramos | Celdas nuevas (objetivo "asistentes avanzados") |
| **D** | Interfaz Gradio v2 con imágenes en el chat | Reemplaza la celda **14 · Gradio** |
| **E** | Benchmark reproducible para el informe final | Celda nueva, al final |

**Requisitos previos (del v7, ya ejecutados):** `groq_client`, `collection`, `buscar_chunks()`,
`DRIVE_DIR`. No hace falta instalar nada nuevo: `sympy`, `numpy` y `matplotlib` ya vienen en Colab.

**Cómo migrar:** ejecutar el v7 hasta la celda *10 · Retrieval* inclusive → luego correr las
celdas de este notebook en orden (A → B → C → D). La sección E se corre cuando quieran
generar los datos del informe.


## 1 · Instalación

In [ ]:
!pip install -q sentence-transformers chromadb groq gradio
print('✅ Dependencias instaladas')

## 2 · Imports

In [ ]:
import json, os, time, shutil
from collections import Counter
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
print('✅ Imports OK')

## 3 · API de GROQ



In [ ]:
from google.colab import userdata
from groq import Groq

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('✅ Groq configurado')
print('   Modelo: llama-3.3-70b-versatile')
print('   Quota: ~14.400 requests/día (gratuito)')

## 4 · Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_DIR    = '/content/drive/MyDrive/PROYECTO_FISICA'
DRIVE_CHROMA = f'{DRIVE_DIR}/chroma_db'
LOCAL_CHROMA = '/content/chroma_fisica'

os.makedirs(DRIVE_DIR, exist_ok=True)

# Copia Drive → local SOLO si no existe ya una copia local
# (evita re-copiar innecesariamente si se re-ejecuta la celda)
if not os.path.exists(LOCAL_CHROMA) and os.path.exists(DRIVE_CHROMA):
    print('📂 Copiando ChromaDB de Drive → local...')
    shutil.copytree(DRIVE_CHROMA, LOCAL_CHROMA)
    print('✅ Copia lista')
elif os.path.exists(LOCAL_CHROMA):
    print('✅ ChromaDB local ya existe — no se re-copia')
else:
    os.makedirs(LOCAL_CHROMA, exist_ok=True)
    print('ℹ️  Sin DB previa en Drive. Se creará una nueva.')

print(f'   Drive : {DRIVE_DIR}')
print(f'   Chroma: {LOCAL_CHROMA} (local)')

## 5 · Modelo de embeddings local

Se descarga una sola vez (~1 GB). En sesiones siguientes se carga desde caché de Colab.
No consume ninguna quota de API.

In [ ]:
# HF_HOME en Drive: el modelo se descarga UNA sola vez (~1 GB)
# y en sesiones siguientes carga desde ahí en ~15 segundos
os.environ['HF_HOME'] = f'{DRIVE_DIR}/hf_cache'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)

print('Cargando modelo de embeddings...')
print('Primera vez: ~2 min (descarga a Drive) | Siguientes: ~15 s (desde caché)')

MODELO_NOMBRE = 'intfloat/multilingual-e5-base'
modelo_embed  = SentenceTransformer(MODELO_NOMBRE)

print(f'✅ Modelo cargado: {MODELO_NOMBRE}')
print(f'   Dimensión de embeddings: {modelo_embed.get_embedding_dimension()}')

## 6 · Funciones de embedding (locales)

In [ ]:
# multilingual-e5 requiere un prefijo según el uso
# 'query: ' para búsquedas, 'passage: ' para documentos
PREFIJO_QUERY   = 'query: '
PREFIJO_DOC     = 'passage: '

_embed_cache = {}

def embed_single(texto: str) -> list:
    """Embedding para BÚSQUEDA (query). Local, sin API."""
    clave = 'q:' + texto
    if clave in _embed_cache:
        return _embed_cache[clave]
    emb = modelo_embed.encode(
        PREFIJO_QUERY + texto,
        normalize_embeddings=True
    ).tolist()
    _embed_cache[clave] = emb
    return emb

def embed_batch(textos: list) -> list:
    """Embedding para INDEXACIÓN (documentos). Local, sin API."""
    con_prefijo = [PREFIJO_DOC + t for t in textos]
    embs = modelo_embed.encode(
        con_prefijo,
        normalize_embeddings=True,
        batch_size=64,
        show_progress_bar=False
    )
    return embs.tolist()

# Test rápido
test_emb = embed_single('aceleración centrípeta')
print(f'✅ Embedding de prueba OK — dimensión: {len(test_emb)}')

## 7 · Carga y normalización de chunks

In [ ]:
CHUNK_FILES = [
    f'{DRIVE_DIR}/physics_chunks1.json',
    f'{DRIVE_DIR}/physics_chunks2.json',
    f'{DRIVE_DIR}/physics_chunks3.json',
    f'{DRIVE_DIR}/ejercicios_adicionales.json',
    f'{DRIVE_DIR}/ejercicios_resueltos_chunks.json',
    f'{DRIVE_DIR}/guia2_enunciados_chunks.json',
    f'{DRIVE_DIR}/guia2_chunks.json',
    f'{DRIVE_DIR}/guia3_chunks.json',
    f'{DRIVE_DIR}/guia3_enunciados_chunks.json',
    f'{DRIVE_DIR}/teoria_guia3_chunks.json',
]

def norm_ej_num(raw) -> str:
    # '009' → '9', None → ''
    if not raw or str(raw).lower() in ('', 'none', 'detectar en texto'):
        return ''
    try:
        return str(int(str(raw)))
    except ValueError:
        return str(raw).strip()

raw_chunks = []
for fpath in CHUNK_FILES:
    if os.path.exists(fpath):
        with open(fpath, encoding='utf-8') as f:
            data = json.load(f)
        raw_chunks.extend(data)
        print(f'  ✓ {os.path.basename(fpath):45s} → {len(data)} chunks')
    else:
        print(f'  ⚠️  No encontrado: {fpath}')

MIN_CONTENT  = 30

chunks = []
for c in raw_chunks:
    content = c.get('content','').strip()
    if len(content) < MIN_CONTENT:
        continue
    meta = c.get('metadata', {})
    chunks.append({
        'content': content,
        'metadata': {
            'fuente':           str(meta.get('fuente','desconocida')),
            'tipo':             str(meta.get('tipo','teoria')),
            'numero_ejercicio': norm_ej_num(meta.get('numero_ejercicio') or
                                meta.get('ejercicio_num', '')),
            'tema':             str(meta.get('tema','')),
            'subtema':          str(meta.get('subtema','')),
            'seccion': str(meta.get('seccion', '')),
            'parte':   str(meta.get('parte', '')),
            'tags':             str(meta.get('tags','')),
        }
    })

print(f'\nTotal chunks listos: {len(chunks)}')
print('\nPor fuente:')
for k,v in Counter(c['metadata']['fuente'] for c in chunks).most_common():
    marca = ' ⭐' if k == 'ejercicios_resueltos' else ''
    print(f'  {v:5d}  {k}{marca}')

## 8 · ChromaDB

In [ ]:
chroma_client = chromadb.PersistentClient(path=LOCAL_CHROMA)
collection    = chroma_client.get_or_create_collection(
    name='fisica',
    metadata={'hnsw:space': 'cosine'}
)
print(f'✅ Collection lista. Documentos actuales: {collection.count()}')

## 9 · Indexación

Sin quota de API — corre tan rápido como la GPU/CPU de Colab lo permita.  
Tiempo estimado: **2–4 minutos** para 2574 chunks en CPU, menos en GPU.

In [ ]:
BATCH_SIZE = 128  # mayor que antes porque no hay rate limit

ya_indexados = collection.count()

if ya_indexados >= len(chunks):
    print(f'✅ Ya indexado ({ya_indexados} docs). Saltando.')
    print('   Para re-indexar: borrá chroma_db de Drive y reiniciá el runtime.')
else:
    if ya_indexados > 0:
        ids_existentes = set(collection.get(include=[])['ids'])
        print(f'ℹ️  DB parcial ({ya_indexados} docs). Continuando...')
    else:
        ids_existentes = set()

    pendientes = [(i,c) for i,c in enumerate(chunks)
                  if f'doc_{i}' not in ids_existentes]

    print(f'Indexando {len(pendientes)} chunks (embeddings locales)...')
    t0 = time.time()

    for start in range(0, len(pendientes), BATCH_SIZE):
        batch       = pendientes[start:start+BATCH_SIZE]
        textos_b    = [c['content']  for _,c in batch]
        metadatas_b = [c['metadata'] for _,c in batch]
        ids_b       = [f'doc_{i}'    for i,_ in batch]
        embeddings_b = embed_batch(textos_b)
        collection.upsert(
            documents=textos_b, embeddings=embeddings_b,
            metadatas=metadatas_b, ids=ids_b
        )
        pct = (start+len(batch))/len(pendientes)*100
        print(f'  ✓ {start+len(batch):5d}/{len(pendientes)} ({pct:.0f}%)')

    elapsed = time.time() - t0
    print(f'\n✅ Listo en {elapsed:.0f}s. Docs en ChromaDB: {collection.count()}')

## 10 · Retrieval

In [ ]:
def buscar_chunks(pregunta, k=5, filtro_tipo=None, filtro_seccion=None, filtro_numero=None):
    emb = embed_single(pregunta)
    condiciones = []
    if filtro_tipo:    condiciones.append({'tipo':             filtro_tipo})
    if filtro_seccion: condiciones.append({'seccion':          filtro_seccion})
    if filtro_numero:  condiciones.append({'numero_ejercicio': filtro_numero})

    where = None
    if len(condiciones) == 1:
        where = condiciones[0]
    elif len(condiciones) > 1:
        where = {'$and': condiciones}

    kwargs = dict(query_embeddings=[emb], n_results=k,
                  include=['documents','metadatas','distances'])
    if where:
        kwargs['where'] = where

    try:
        r = collection.query(**kwargs)
    except Exception:
        # Si el filtro no matchea nada, busca sin filtro
        r = collection.query(query_embeddings=[emb], n_results=k,
                             include=['documents','metadatas','distances'])

    return [{'content': d, 'metadata': m, 'score': round(1-dist, 3)}
            for d, m, dist in zip(r['documents'][0],
                                  r['metadatas'][0],
                                  r['distances'][0])]

# Test
test = buscar_chunks('aceleración en caída libre', k=3)
print('Test de búsqueda — top 3:\n')
for r in test:
    m = r['metadata']
    print(f"  [{r['score']}] {m['fuente']} | {m['tipo']}")
    print(f"  {r['content'][:150]}\n")

## A · Precisión numérica — «el LLM piensa, Python calcula»

**Problema (benchmark del informe de avance):** P2 y P8 fallaron por redondeo/aritmética,
P5 por confundir el fin de una fase con el máximo real, T2 por un error conceptual de MCU
y en P1 no se detectó la inconsistencia de datos.

**Solución — pipeline de dos pasadas:**

1. **Resolver:** el LLM hace el IPEE pero en *E-Ejecutar* despeja en forma **simbólica** y
   delega TODA la aritmética en un bloque ` ```python # CALC``` ` (con `math`, `numpy`,
   `sympy` y `g = 9.8` precargados).
2. **Calcular:** el bloque se ejecuta acá, en un entorno restringido (sin imports, sin
   archivos). Los `print()` devuelven los valores **exactos**.
3. **Redactar:** una segunda llamada reescribe la respuesta final usando *esos* números y
   responde una **checklist física** ítem por ítem (consistencia de datos, fases del
   movimiento, signos, vínculo de rozamiento, conceptos de circular).

Cada error del benchmark tiene su contramedida: la aritmética la hace Python (P2, P8),
la checklist obliga a preguntarse si el movimiento continúa después de la fase (P5) y si
los datos cierran (P1), y el prompt fija los conceptos de MCU (T2).

In [ ]:
# ============================================================
#  EJECUTOR DE CÁLCULOS — v1
#  El LLM escribe la resolución simbólica y delega la aritmética
#  en un bloque ```python``` que acá se ejecuta en un entorno
#  restringido (math + numpy + sympy). La salida (prints) vuelve
#  al LLM para redactar la respuesta final con cifras EXACTAS.
# ============================================================
import io, re, math, contextlib
import numpy as np
import sympy as sp

G_DEFAULT = 9.8  # m/s² — valor usado por la cátedra (cambiar si corresponde)

_RE_BLOQUE = re.compile(r'```(?:python|py)?\s*\n(.*?)```', re.DOTALL)
_MODULOS_OK = ('numpy', 'sympy', 'math')

_BUILTINS_OK = {n: getattr(__builtins__, n) if not isinstance(__builtins__, dict)
                else __builtins__[n]
                for n in ('abs', 'min', 'max', 'round', 'range', 'len', 'print',
                          'enumerate', 'zip', 'float', 'int', 'str', 'bool',
                          'sum', 'sorted', 'list', 'tuple', 'dict', 'pow')}


def extraer_bloque_calc(texto):
    """Devuelve el código de los bloques ```python``` (concatenados) o None."""
    bloques = _RE_BLOQUE.findall(texto or '')
    if not bloques:
        return None
    con_marca = [b for b in bloques if '# CALC' in b or '#CALC' in b]
    return '\n\n'.join(con_marca if con_marca else bloques)


def _limpiar_codigo(codigo):
    """Permite solo imports de math/numpy/sympy (ya precargados); bloquea el resto."""
    lineas = []
    for ln in codigo.splitlines():
        s = ln.strip()
        if s.startswith(('import ', 'from ')):
            if any(m in s for m in _MODULOS_OK):
                continue                      # ya están precargados
            raise ValueError(f'Import no permitido: "{s}"')
        if '__' in s or 'open(' in s or 'eval(' in s or 'exec(' in s:
            raise ValueError(f'Instrucción no permitida: "{s}"')
        lineas.append(ln)
    return '\n'.join(lineas)


def ejecutar_calculos(texto_o_codigo, g=G_DEFAULT, max_chars=2500, es_codigo=False):
    """
    Ejecuta el bloque CALC. Devuelve dict:
      {'ok': bool, 'codigo': str|None, 'salida': str, 'error': str|None}
    Por defecto extrae el bloque ```python``` del texto; con es_codigo=True
    ejecuta el string tal cual.
    """
    codigo = texto_o_codigo if es_codigo else extraer_bloque_calc(texto_o_codigo)
    if not codigo or not codigo.strip():
        return {'ok': False, 'codigo': None, 'salida': '', 'error': 'sin_bloque'}

    try:
        codigo = _limpiar_codigo(codigo)
    except ValueError as e:
        return {'ok': False, 'codigo': codigo, 'salida': '', 'error': str(e)}

    ns = {
        '__builtins__': _BUILTINS_OK,
        'np': np, 'numpy': np, 'sp': sp, 'sympy': sp, 'math': math,
        'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos, 'tan': math.tan,
        'asin': math.asin, 'acos': math.acos, 'atan': math.atan,
        'atan2': math.atan2, 'radians': math.radians, 'degrees': math.degrees,
        'pi': math.pi, 'exp': math.exp, 'log': math.log,
        'g': g, 'G': g,
    }
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            exec(compile(codigo, '<calc>', 'exec'), ns)   # noqa: S102 — namespace restringido
        salida = buf.getvalue().strip()
        if len(salida) > max_chars:
            salida = salida[:max_chars] + '\n[...salida truncada...]'
        if not salida:
            return {'ok': False, 'codigo': codigo, 'salida': '',
                    'error': 'el bloque no imprimió nada (faltan print())'}
        return {'ok': True, 'codigo': codigo, 'salida': salida, 'error': None}
    except Exception as e:
        return {'ok': False, 'codigo': codigo, 'salida': buf.getvalue().strip(),
                'error': f'{type(e).__name__}: {e}'}

In [ ]:
# ============================================================
#  PIPELINE v2 — «resolver → calcular en Python → redactar»
#  Reemplaza a SYSTEM_PROMPT y ask_physics() de la versión 7.
#  Ataca directamente los errores del benchmark:
#    P2/P8 (aritmética/redondeo)  → la calcula Python, no el LLM
#    P5   (fases del movimiento)  → checklist obligatoria
#    T2   (concepto MCU)          → guardarraíl en el prompt
#    P1   (datos inconsistentes)  → checklist obligatoria
# ============================================================
import re, time

MODELO_LLM    = 'llama-3.3-70b-versatile'   # el "capo": razona (primera pasada)
MODELO_RAPIDO = 'llama-3.1-8b-instant'      # el "veloz": tareas mecánicas

SYSTEM_PROMPT_V2 = """
Sos un tutor de Física I universitaria (Mecánica: cinemática, dinámica, trabajo y
energía, cantidad de movimiento y momento angular). Explicás y resolvés problemas
con rigor y claridad usando el método IPEE. Respondé siempre en español.

REGLAS FUNDAMENTALES
1. Usá EXCLUSIVAMENTE la información del CONTEXTO proporcionado.
2. Si el contexto no alcanza, decilo claramente. No inventes datos ni fórmulas.
3. Citá la fuente entre corchetes cuando uses el contexto, p. ej. [Young & Freedman].
4. Las fórmulas siempre en LaTeX ($...$ o $$...$$), nunca en texto plano.
5. Si la consulta es trivial o no es de física, respondé breve y directo, SIN IPEE.

PRECISIÓN NUMÉRICA — OBLIGATORIO en ejercicios con números
- NO hagas aritmética "a mano": el cálculo lo hace Python.
- En el paso E-Ejecutar: primero despejá la incógnita en forma SIMBÓLICA y luego
  escribí UN ÚNICO bloque de código así:

```python
# CALC
# datos (todo en unidades del SI; conversiones acá adentro)
v0 = 70/3.6          # km/h -> m/s
mu_s = 0.3
# fórmulas ya despejadas, en orden
a_max = mu_s*g
d_min = v0**2/(2*a_max)
print(f"v0    = {v0:.4f} m/s")
print(f"a_max = {a_max:.4f} m/s^2")
print(f"d_min = {d_min:.4f} m")
```

- Ya están disponibles: math, np (numpy), sp (sympy), funciones sqrt/sin/cos/tan/
  atan2/radians/degrees, pi, y la constante g = 9.8 m/s². NO uses import.
- print() de TODOS los resultados intermedios y finales, con nombre y unidad.
- No redondees a mano ni uses valores intermedios redondeados: dejá que Python
  arrastre la precisión completa.

MÉTODO IPEE
I — Identificar: datos (símbolo, valor, unidad), incógnitas, tipo de movimiento y,
    si corresponde, las FASES del movimiento (¿qué pasa antes/durante/después?).
P — Plantear: sistema de referencia y ejes elegidos (y POR QUÉ conviene ese sistema),
    fuerzas actuantes si es dinámica, ecuaciones aplicables, condiciones de vínculo,
    y si se conserva alguna cantidad (energía, p, L) justificando por qué.
E — Ejecutar: despeje simbólico + bloque # CALC.
E — Evaluar: unidades, orden de magnitud, casos límite y la CHECKLIST completa.

CHECKLIST FÍSICA (respondela SIEMPRE, ítem por ítem, en E-Evaluar)
1. Consistencia: ¿los datos del enunciado son coherentes entre sí? Si algo no
   cierra o sobra un dato contradictorio, SEÑALALO explícitamente.
2. Fases: ¿la magnitud pedida ocurre dentro de la fase analizada, o el movimiento
   CONTINÚA después? (ej.: al cortarse un cable, el cuerpo sigue subiendo mientras
   v > 0; el máximo real puede estar en la fase siguiente).
3. Signos: ¿coherentes con el sistema de ejes declarado en P?
4. Vínculos y rozamiento: f_s ≤ μ_s·N (el igual SOLO en el límite de deslizamiento);
   la fricción estática se opone al deslizamiento RELATIVO que tendería a ocurrir.
5. Conceptos de movimiento circular: la velocidad es TANGENTE a la trayectoria
   (nunca apunta al centro); la aceleración centrípeta a_n = v²/R apunta AL CENTRO;
   en MCU a_t = 0 y |v| es constante aunque v cambia de dirección.

FORMATO
- Markdown con títulos ##, negritas y listas.
- Fórmulas destacadas en $$...$$.
- Antes de cerrar: **Resultado:** [valor con unidades].
- Al final: **Error frecuente:** [el más común en este tipo de problema].
""".strip()

PROMPT_REDACTOR = """
Sos el mismo tutor de Física I. Recibís tu BORRADOR y la SALIDA VERIFICADA que
produjo Python al ejecutar tu bloque # CALC. Esos números son EXACTOS y mandan
sobre cualquier número del borrador.

Redactá la RESPUESTA FINAL para el estudiante:
1. Mantené la estructura IPEE del borrador pero SIN el bloque de código: mostrá el
   despeje simbólico y a continuación los valores numéricos verificados.
2. Reemplazá TODO valor numérico por el verificado. Redondeá a 3–4 cifras
   significativas SOLO al presentar (aclarando el valor de g usado).
3. En E-Evaluar respondé la checklist ítem por ítem (consistencia de datos, fases
   del movimiento, signos, vínculos/rozamiento, conceptos de circular si aplica).
4. Cerrá con **Resultado:** y **Error frecuente:**.
5. Todo en español, fórmulas en LaTeX, citando fuentes del contexto si las usaste.
""".strip()


def _llm(messages, temperature=0.1, max_tokens=1500, modelo=None, reintentos_rate=4):
    modelo = modelo or MODELO_LLM
    for intento in range(reintentos_rate + 1):
        try:
            r = groq_client.chat.completions.create(
                model=modelo, messages=messages,
                temperature=temperature, max_tokens=max_tokens,
            )
            return r.choices[0].message.content
        except Exception as e:
            msg = str(e)
            es_limite = '429' in msg or 'rate_limit' in msg.lower()
            if es_limite and intento < reintentos_rate:
                m = re.search(r'try again in ([\d.]+)s', msg)
                espera = (float(m.group(1)) + 1.5) if m else 20
                print(f'⏳ Cupo de Groq lleno. Espero {espera:.0f}s y reintento… '
                      f'({intento+1}/{reintentos_rate})')
                time.sleep(espera)
                continue
            raise


# ---------- recuperación de contexto (igual espíritu que v7, factorizada) ----------
import re as _re
_EJ_PATTERN = _re.compile(r'\b(?:ejercicio|ej\.?)\s*(\d+)\b', _re.IGNORECASE)


def _detectar_num_ej(pregunta):
    m = _EJ_PATTERN.search(pregunta)
    return str(int(m.group(1))) if m else None


def _dedup(lista, max_k=10):
    seen = {}
    for c in lista:
        key = c['content'][:100]
        if key not in seen or c['score'] > seen[key]['score']:
            seen[key] = c
    return sorted(seen.values(), key=lambda x: -x['score'])[:max_k]


def recuperar_contexto(pregunta, k=5, max_k=10):
    recuperados = buscar_chunks(pregunta, k=k)
    num_ej = _detectar_num_ej(pregunta)
    if num_ej:
        recuperados = buscar_chunks(pregunta, k=6, filtro_numero=num_ej) + recuperados
    try:  # priorizar resoluciones propias paso a paso
        recuperados += buscar_chunks(pregunta, k=3, filtro_tipo='ejercicio')
    except Exception:
        pass
    return _dedup(recuperados, max_k=max_k)


def _bloque_contexto(chunks):
    bloques = []
    for i, ch in enumerate(chunks, 1):
        m = ch['metadata']
        partes = [m.get('fuente', '?')]
        if m.get('numero_ejercicio'):
            partes.append(f"Ej.{m['numero_ejercicio']}")
        partes.append(m.get('tipo', '?'))
        bloques.append(f"--- Fragmento {i} [{' - '.join(partes)}] ---\n{ch['content']}")
    return '\n\n'.join(bloques)


# ---------- pipeline principal ----------
ULTIMA_TRAZA = {}   # para debug y para medir tiempos en el informe


def ask_physics_v2(pregunta, k=5, verificar=True, mostrar_codigo=False,
                   mostrar_chunks=False):
    """
    Pipeline «resolver → calcular en Python → redactar».
      verificar=False      → una sola pasada (comportamiento tipo v7, más rápido)
      mostrar_codigo=True  → agrega al final el bloque CALC y su salida verificada
    """
    global ULTIMA_TRAZA
    t0 = time.time()
    chunks = recuperar_contexto(pregunta, k=k)
    t_ret = time.time() - t0

    if mostrar_chunks:
        for i, c in enumerate(chunks, 1):
            m = c['metadata']
            print(f"[{i}] {m['fuente']} {('Ej.'+m['numero_ejercicio']) if m['numero_ejercicio'] else ''} | score={c['score']}")

    user_msg = f"CONTEXTO:\n{_bloque_contexto(chunks)}\n\nPREGUNTA: {pregunta}\n\nRESPUESTA:"

    t1 = time.time()
    try:
        borrador = _llm([{'role': 'system', 'content': SYSTEM_PROMPT_V2},
                         {'role': 'user', 'content': user_msg}])
    except Exception as e:
        return f'❌ Error Groq: {e}'
    t_llm1 = time.time() - t1

    calc = ejecutar_calculos(borrador)
    # si el código falló, un único reintento pidiendo la corrección
    if calc['error'] not in (None, 'sin_bloque') and verificar:
        try:
            fix = _llm([{'role': 'system', 'content': SYSTEM_PROMPT_V2},
                        {'role': 'user', 'content':
                         f"Tu bloque # CALC falló con este error:\n{calc['error']}\n\n"
                         f"Código:\n```python\n{calc['codigo']}\n```\n\n"
                         "Devolvé SOLO el bloque ```python``` corregido, nada más."}],
                       max_tokens=900)
            calc = ejecutar_calculos(fix)
        except Exception:
            pass

    if not verificar or calc['error'] == 'sin_bloque':
        # pregunta teórica (o modo rápido): el borrador ES la respuesta
        respuesta, t_llm2 = borrador, 0.0
        if calc['error'] not in (None, 'sin_bloque'):
            respuesta += '\n\n> ⚠ *Verificación numérica no disponible en esta respuesta.*'
    else:
        t2 = time.time()
        salida = calc['salida'] if calc['ok'] else f"(el código falló: {calc['error']})"
        try:
            respuesta = _llm([
                {'role': 'system', 'content': PROMPT_REDACTOR},
                {'role': 'user', 'content':
                 f"BORRADOR:\n{borrador}\n\n"
                 f"SALIDA VERIFICADA DE PYTHON:\n{salida}\n\n"
                 f"PREGUNTA ORIGINAL: {pregunta}"}])
        except Exception as e:
            respuesta = borrador + f'\n\n> ⚠ *No se pudo redactar la versión verificada: {e}*'
        t_llm2 = time.time() - t2
        if not calc['ok']:
            respuesta += '\n\n> ⚠ *La verificación numérica falló; revisar los valores.*'

    if mostrar_codigo and calc.get('codigo'):
        respuesta += (f"\n\n---\n<details><summary>🔍 Verificación numérica (Python)</summary>\n\n"
                      f"```python\n{calc['codigo'].strip()}\n```\n\n"
                      f"**Salida:**\n```\n{calc.get('salida','')}\n```\n</details>")

    ULTIMA_TRAZA = {'pregunta': pregunta, 'chunks': chunks, 'borrador': borrador,
                    'calc': calc, 't_retrieval': round(t_ret, 2),
                    't_llm_borrador': round(t_llm1, 2),
                    't_llm_redaccion': round(t_llm2, 2),
                    't_total': round(time.time() - t0, 2)}
    return respuesta


print('✅ ask_physics_v2() lista  —  ej.: print(ask_physics_v2("Resolvé el ejercicio 12"))')

## B · Diagrama de Cuerpo Libre (DCL)

**Diseño:** el LLM **no dibuja** — describe el diagrama como un **JSON estructurado**
(fuerzas con ángulo y magnitud relativa, sistema de ejes elegido y su justificación,
vectores $\vec v$, $\vec a$, $\vec r$) y una función determinística de `matplotlib`
lo renderiza siempre con el mismo estilo. Esto es mucho más confiable que pedirle al
modelo que escriba código de dibujo.

Qué sale en la figura:

* **Panel izquierdo (DCL):** solo las fuerzas reales sobre el cuerpo, con sus
  **proyecciones en los ejes** cuando no están alineadas (ej.: $mg\sin\theta$ y
  $mg\cos\theta$ en el plano inclinado), la superficie (piso / plano inclinado / techo),
  soga si la hay, y los **ejes del sistema elegido**: cartesianos (rotables), polares
  ($\hat r$, $\hat\theta$) o intrínsecos ($\hat t$, $\hat n$ + centro de curvatura).
* **Panel derecho (esquema cinemático):** $\vec v$, $\vec a$ (y $\vec r$ en polares)
  separados de las fuerzas — así el DCL queda "puro", como pide la cátedra.
* Al pie: la **justificación de los ejes** y hasta 3 notas pedagógicas.

In [ ]:
# ============================================================
#  RENDERIZADOR DE DCL (Diagrama de Cuerpo Libre) — v1
#  Dibuja un DCL a partir de una especificación JSON generada
#  por el LLM. Soporta ejes cartesianos / polares / intrínsecos,
#  fuerzas, vectores cinemáticos (v, a, r) y sus proyecciones.
# ============================================================
import json, re, os, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch, Arc, Circle, Polygon

# ---------- paleta ----------
COLOR_FUERZAS = ['#c62828', '#e65100', '#6a1b9a', '#ad1457', '#4e342e', '#b71c1c']
COLOR_CIN = {'velocidad': '#1565c0', 'aceleracion': '#2e7d32', 'posicion': '#00838f'}
COLOR_EJES = '#455a64'
COLOR_CUERPO = '#90a4ae'
COLOR_SUPERF = '#78909c'

_D = np.radians  # grados → radianes

_MATH_FIX = {r'\le ': r'\leq ', r'\le$': r'\leq$', r'\ge ': r'\geq ', r'\ge$': r'\geq$',
             r'\ne ': r'\neq ', r'\to': r'\rightarrow', r'\text': r'\mathrm',
             r'\dfrac': r'\frac', r'\;': r'\ ', r'\,': r'\ '}


def _texto_seguro(s):
    """Sanea LaTeX generado por el LLM para que mathtext no rompa la figura."""
    if not s:
        return ''
    s = str(s)
    for k, v in _MATH_FIX.items():
        s = s.replace(k, v)
    if s.count('$') % 2 == 1:                      # $ desbalanceado
        s = s.replace('$', '')
    try:
        from matplotlib.mathtext import MathTextParser
        MathTextParser('agg').parse(s)
        return s
    except Exception:
        # último recurso: quitar todo el modo matemático
        s = s.replace('$', '')
        s = re.sub(r'\\[a-zA-Z]+', '', s).replace('{', '').replace('}', '')
        return s


def _unit(ang_deg):
    a = _D(ang_deg)
    return np.array([np.cos(a), np.sin(a)])


def _fmt_label(nombre):
    """Convierte el nombre a etiqueta matplotlib (mathtext si tiene LaTeX)."""
    if nombre is None:
        return ''
    s = str(nombre).strip().strip('$')
    if any(t in s for t in ('\\', '_', '^', '{')):
        return f'${s}$'
    return s


def _base_name(nombre):
    """'\\vec{f}_s' → 'f_s'  (para etiquetas de componentes)."""
    s = str(nombre).strip().strip('$')
    s = re.sub(r'\\vec\{([^}]*)\}', r'\1', s)
    s = re.sub(r'\\hat\{([^}]*)\}', r'\1', s)
    s = s.split('=')[0].strip()
    return s


def _flecha(ax, origen, vec, color, lw=2.4, estilo='-', z=5, hueco=False):
    """Dibuja una flecha con punta triangular. vec = np.array [dx,dy]."""
    fin = np.asarray(origen) + np.asarray(vec)
    kw = dict(arrowstyle='-|>', mutation_scale=20, lw=lw,
              color=color, linestyle=estilo, zorder=z,
              shrinkA=0, shrinkB=0)
    if hueco:
        kw.update(arrowstyle='-|>', facecolor='white')
    ax.add_patch(FancyArrowPatch(origen, fin, **kw))
    return fin


def _etiqueta(ax, pos, texto, color, ang_deg=None, dist=0.14, fs=13, z=8):
    """Etiqueta cerca de `pos`, desplazada en la dirección ang_deg."""
    off = _unit(ang_deg) * dist if ang_deg is not None else np.array([dist, dist])
    p = np.asarray(pos) + off
    ax.text(p[0], p[1], _texto_seguro(texto), color=color, fontsize=fs, zorder=z,
            ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.12', fc='white', ec='none', alpha=0.75))


def _dibujar_cuerpo(ax, tipo='bloque', rot_deg=0.0, alpha=1.0):
    """Bloque (cuadrado, rotado con la superficie) o partícula (punto)."""
    if tipo == 'particula':
        ax.add_patch(Circle((0, 0), 0.07, fc='#37474f', ec='black', zorder=6, alpha=alpha))
    else:
        l = 0.42
        base = np.array([[-l, -l], [l, -l], [l, l], [-l, l]]) / 2 * 1.6
        a = _D(rot_deg)
        R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
        pts = base @ R.T
        ax.add_patch(Polygon(pts, closed=True, fc=COLOR_CUERPO, ec='#263238',
                             lw=1.6, zorder=4, alpha=alpha))
    ax.plot(0, 0, 'o', color='black', ms=4, zorder=7, alpha=alpha)


def _dibujar_superficie(ax, sup, lado_cuerpo=0.34):
    """Suelo horizontal, plano inclinado o techo. El cuerpo queda en (0,0)."""
    if not sup:
        return 0.0
    tipo = str(sup.get('tipo', 'ninguna')).lower()
    ang = float(sup.get('angulo', 0) or 0)
    if tipo.startswith('horiz'):
        y0 = -lado_cuerpo
        ax.plot([-2.0, 2.0], [y0, y0], color=COLOR_SUPERF, lw=2.5, zorder=2)
        for x in np.arange(-1.9, 2.0, 0.25):          # rayado del piso
            ax.plot([x, x - 0.13], [y0, y0 - 0.13], color=COLOR_SUPERF, lw=1, zorder=2)
        return 0.0
    if tipo.startswith('techo'):
        y0 = +1.55
        ax.plot([-2.0, 2.0], [y0, y0], color=COLOR_SUPERF, lw=2.5, zorder=2)
        for x in np.arange(-1.9, 2.0, 0.25):
            ax.plot([x, x + 0.13], [y0, y0 + 0.13], color=COLOR_SUPERF, lw=1, zorder=2)
        return 0.0
    if tipo.startswith('inclin'):
        a = _D(ang)
        t_hat = np.array([np.cos(a), np.sin(a)])      # a lo largo del plano
        n_hat = np.array([-np.sin(a), np.cos(a)])     # normal al plano
        p0 = -n_hat * lado_cuerpo                     # punto de apoyo bajo el cuerpo
        A = p0 - t_hat * 1.9                          # pie del plano (izquierda-abajo)
        B = p0 + t_hat * 1.4                          # cima
        C = np.array([B[0], A[1]])                    # vértice recto
        ax.add_patch(Polygon([A, B, C], closed=True, fc='#eceff1',
                             ec=COLOR_SUPERF, lw=2.2, zorder=1))
        # arco del ángulo θ en el pie del plano
        r_arc = 0.55
        ax.add_patch(Arc(A, 2 * r_arc, 2 * r_arc, angle=0, theta1=0, theta2=ang,
                         color='#37474f', lw=1.4, zorder=3))
        mid = _unit(ang / 2) * (r_arc + 0.17)
        ax.text(A[0] + mid[0], A[1] + mid[1], r'$\theta$', fontsize=13,
                ha='center', va='center', color='#37474f')
        return ang
    return 0.0


def _ejes_cartesianos(ax, rot_deg, pos=(1.55, 1.35), L=0.62, sufijo=''):
    """Tríada x–y en una esquina, rotada rot_deg."""
    o = np.asarray(pos)
    e1, e2 = _unit(rot_deg), _unit(rot_deg + 90)
    _flecha(ax, o, e1 * L, COLOR_EJES, lw=1.8, z=3)
    _flecha(ax, o, e2 * L, COLOR_EJES, lw=1.8, z=3)
    _etiqueta(ax, o + e1 * L, f'$x{sufijo}$', COLOR_EJES, rot_deg, 0.14, fs=12)
    _etiqueta(ax, o + e2 * L, f'$y{sufijo}$', COLOR_EJES, rot_deg + 90, 0.14, fs=12)
    ax.plot(*o, 'o', color=COLOR_EJES, ms=3, zorder=3)


def _versores(ax, ang1, ang2, lbl1, lbl2, L=0.5):
    """Versores del sistema (polar / intrínseco) anclados en la partícula."""
    for ang, lbl in ((ang1, lbl1), (ang2, lbl2)):
        fin = _flecha(ax, (0, 0), _unit(ang) * L, COLOR_EJES, lw=1.9, z=9)
        _etiqueta(ax, fin, lbl, COLOR_EJES, ang, 0.15, fs=12)


def _proyectar(ax, origen, vec, ang_e1, color, lbl_base, suf1, suf2,
               etiquetas=None):
    """Dibuja las componentes de `vec` sobre los ejes e1 (ang_e1) y e2 (+90°)."""
    e1, e2 = _unit(ang_e1), _unit(ang_e1 + 90)
    c1, c2 = float(np.dot(vec, e1)), float(np.dot(vec, e2))
    tip = np.asarray(origen) + vec
    for c, e, suf, k in ((c1, e1, suf1, 0), (c2, e2, suf2, 1)):
        if abs(c) < 0.09:            # el vector ya está sobre un eje
            continue
        comp = e * c
        fin = _flecha(ax, origen, comp, color, lw=1.5, estilo='--', z=4)
        # línea punteada de la punta del vector a la punta de la componente
        ax.plot([tip[0], fin[0]], [tip[1], fin[1]], ':', color=color, lw=1.1, zorder=3)
        if etiquetas and k < len(etiquetas) and etiquetas[k]:
            txt = _fmt_label(etiquetas[k])
        else:
            txt = f'${lbl_base}_{{{suf}}}$'
        ang_lbl = np.degrees(np.arctan2(comp[1], comp[0]))
        _etiqueta(ax, fin, txt, color, ang_lbl, 0.18, fs=11)


def dibujar_dcl(spec, ruta_png=None, mostrar=True):
    """
    Dibuja el DCL a partir del dict `spec` (ver ESQUEMA_DCL).
    Devuelve la ruta del PNG generado.
    """
    if isinstance(spec, str):
        spec = json.loads(spec)

    sistema = spec.get('sistema', {}) or {}
    tipo_sis = str(sistema.get('tipo', 'cartesiano')).lower()
    rot = float(sistema.get('rotacion', 0) or 0)
    fuerzas = spec.get('fuerzas', []) or []
    cinematica = spec.get('cinematica', []) or []
    sup = spec.get('superficie', {}) or {}
    cuerpo = spec.get('cuerpo', 'bloque')
    pol = spec.get('polar', {}) or {}
    intr = spec.get('intrinseco', {}) or {}

    dos_paneles = len(cinematica) > 0
    fig, axs = plt.subplots(1, 2 if dos_paneles else 1,
                            figsize=(11.5 if dos_paneles else 6.4, 6.0))
    axs = np.atleast_1d(axs)

    # ---------- normalizar magnitudes ----------
    mags_f = [abs(float(f.get('magnitud', 1) or 1)) for f in fuerzas] or [1]
    esc_f = 1.15 / max(mags_f)
    mags_c = [abs(float(v.get('magnitud', 1) or 1)) for v in cinematica] or [1]
    esc_c = 1.05 / max(mags_c)

    # ejes de descomposición según el sistema elegido
    if tipo_sis.startswith('polar'):
        ang_e1 = float(pol.get('angulo_radial', 90) or 90)
        suf1, suf2 = 'r', r'\theta'
    elif tipo_sis.startswith('intr'):
        ang_e1 = float(intr.get('angulo_tangente', 0) or 0)
        lado = str(intr.get('normal_hacia', 'izquierda')).lower()
        suf1, suf2 = 't', 'n'
        ang_n = ang_e1 + (90 if lado.startswith('izq') else -90)
    else:
        ang_e1 = rot
        suf1, suf2 = 'x', 'y'

    # ================= PANEL 1 · DCL (solo fuerzas) =================
    ax = axs[0]
    ang_sup = _dibujar_superficie(ax, sup)
    rot_cuerpo = ang_sup if str(sup.get('tipo', '')).lower().startswith('inclin') else 0
    _dibujar_cuerpo(ax, cuerpo, rot_cuerpo)

    cuerda = spec.get('cuerda')
    if cuerda:
        angc = float(cuerda.get('angulo', 90) or 90)
        Lc = float(cuerda.get('longitud', 1.35) or 1.35)
        fin = np.asarray(_unit(angc)) * Lc
        ax.plot([0, fin[0]], [0, fin[1]], color='#6d4c41', lw=2, zorder=2)
        ax.plot(*fin, 's', color='#6d4c41', ms=7, zorder=2)

    for i, f in enumerate(fuerzas):
        color = f.get('color') or COLOR_FUERZAS[i % len(COLOR_FUERZAS)]
        ang = float(f.get('angulo', 0) or 0)
        L = abs(float(f.get('magnitud', 1) or 1)) * esc_f
        vec = _unit(ang) * L
        fin = _flecha(ax, (0, 0), vec, color, lw=2.6, z=6)
        _etiqueta(ax, fin, _fmt_label(f.get('nombre', f'F_{i+1}')), color, ang, 0.17)
        if f.get('descomponer'):
            e1 = ang_e1 if not tipo_sis.startswith('intr') else ang_e1
            _proyectar(ax, (0, 0), vec, e1, color, _base_name(f.get('nombre', 'F')),
                       suf1, suf2, f.get('etiquetas_componentes'))

    # ejes del sistema en el panel de fuerzas
    if tipo_sis.startswith('cart'):
        _ejes_cartesianos(ax, rot, sufijo="'" if abs(rot) > 1 else '')
    elif tipo_sis.startswith('polar'):
        _versores(ax, ang_e1, ang_e1 + 90, r'$\hat r$', r'$\hat\theta$')
    elif tipo_sis.startswith('intr'):
        _versores(ax, ang_e1, ang_n, r'$\hat t$', r'$\hat n$')

    ax.set_title('DCL — fuerzas sobre el cuerpo', fontsize=12.5, pad=10)

    # ================= PANEL 2 · esquema cinemático =================
    if dos_paneles:
        ax2 = axs[1]
        _dibujar_superficie(ax2, sup)
        _dibujar_cuerpo(ax2, cuerpo, rot_cuerpo, alpha=0.45)

        # trayectoria de referencia
        if tipo_sis.startswith('intr'):
            R = float(intr.get('radio', 1.25) or 1.25)
            centro = _unit(ang_n) * R
            a0 = np.degrees(np.arctan2(-centro[1], -centro[0]))
            arc = Arc(centro, 2 * R, 2 * R, angle=0, theta1=a0 - 55, theta2=a0 + 55,
                      color='#90a4ae', lw=1.6, linestyle='--', zorder=1)
            ax2.add_patch(arc)
            ax2.plot(*centro, '+', color='#90a4ae', ms=10, zorder=1)
            ax2.text(centro[0] + 0.1, centro[1] + 0.1, 'C', color='#90a4ae', fontsize=11)
        if tipo_sis.startswith('polar'):
            d0 = float(pol.get('distancia_origen', 1.5) or 1.5)
            O = -_unit(ang_e1) * d0
            ax2.plot([O[0], 0], [O[1], 0], '--', color='#90a4ae', lw=1.4, zorder=1)
            ax2.plot(*O, 'o', color='#546e7a', ms=6, zorder=2)
            ax2.text(O[0] - 0.15, O[1] - 0.15, 'O', fontsize=12, color='#546e7a')

        for v in cinematica:
            t = str(v.get('tipo', 'velocidad')).lower()
            color = v.get('color') or COLOR_CIN.get(t, '#1565c0')
            ang = float(v.get('angulo', 0) or 0)
            L = abs(float(v.get('magnitud', 1) or 1)) * esc_c
            if t == 'posicion' and tipo_sis.startswith('polar'):
                origen = -_unit(ang_e1) * float(pol.get('distancia_origen', 1.5) or 1.5)
                vec = -np.asarray(origen)
            else:
                origen, vec = (0, 0), _unit(ang) * L
            fin = _flecha(ax2, origen, vec, color, lw=2.6, z=6)
            _etiqueta(ax2, fin, _fmt_label(v.get('nombre', t[0])), color, ang, 0.17)
            if v.get('descomponer'):
                _proyectar(ax2, origen, vec, ang_e1, color,
                           _base_name(v.get('nombre', t[0])), suf1, suf2,
                           v.get('etiquetas_componentes'))

        if tipo_sis.startswith('cart'):
            _ejes_cartesianos(ax2, rot, sufijo="'" if abs(rot) > 1 else '')
        elif tipo_sis.startswith('polar'):
            _versores(ax2, ang_e1, ang_e1 + 90, r'$\hat r$', r'$\hat\theta$')
        elif tipo_sis.startswith('intr'):
            _versores(ax2, ang_e1, ang_n, r'$\hat t$', r'$\hat n$')
        ax2.set_title('Esquema cinemático', fontsize=12.5, pad=10)

    # ---------- estética común ----------
    for a in axs:
        a.set_xlim(-2.15, 2.15); a.set_ylim(-2.05, 2.05)
        a.set_aspect('equal'); a.axis('off')

    titulo = _texto_seguro(spec.get('titulo', 'Diagrama de cuerpo libre'))
    fig.suptitle(titulo, fontsize=14, y=0.99)

    notas = spec.get('notas') or []
    just = (sistema.get('justificacion') or '').strip()
    pie = []
    if just:
        pie.append('Ejes: ' + _texto_seguro(just))
    pie += [f'• {_texto_seguro(n)}' for n in notas[:3]]
    if pie:
        fig.text(0.5, 0.015, '\n'.join(pie), ha='center', fontsize=9.5,
                 color='#37474f', wrap=True)

    fig.tight_layout(rect=[0, 0.06 if pie else 0.01, 1, 0.95])

    if ruta_png is None:
        os.makedirs('figs', exist_ok=True)
        ruta_png = f'figs/dcl_{int(time.time()*1000)}.png'
    fig.savefig(ruta_png, dpi=130, bbox_inches='tight', facecolor='white')
    if mostrar:
        plt.show()
    plt.close(fig)
    return ruta_png

In [ ]:
# ============================================================
#  GENERACIÓN DE DCL — el LLM describe, Python dibuja
#  El modelo NO dibuja: emite un JSON estructurado (fuerzas,
#  ejes, vectores) y dibujar_dcl() lo renderiza siempre igual.
# ============================================================
PROMPT_DCL = """
Sos un experto en Física I. Tu única tarea: a partir del CONTEXTO y la PREGUNTA
(y la RESOLUCIÓN si se incluye), describir el Diagrama de Cuerpo Libre del cuerpo
analizado como UN OBJETO JSON. No escribas nada fuera del JSON (sin ``` ni texto).

CONVENCIONES
- Ángulos en GRADOS, medidos desde la horizontal +x, positivos antihorario
  (90 = hacia arriba, -90 = hacia abajo, 180 = hacia la izquierda).
- "magnitud": tamaño RELATIVO de la flecha entre 0.2 y 1.5 (no hace falta escala real,
  pero respetá relaciones obvias: en equilibrio vertical N ≈ P; en un plano
  inclinado N = P·cosθ < P).
- "nombre": en LaTeX SIN signos $ (ej.: "\\\\vec{N}", "\\\\vec{P}=m\\\\vec{g}", "\\\\vec{f}_s").
- Incluí TODAS las fuerzas reales sobre el cuerpo (peso, normal, tensión, fricción,
  aplicadas...) y NINGUNA fuerza ficticia ni "fuerza del movimiento".
- "descomponer": true solo para vectores que NO están alineados con los ejes elegidos.
- "sistema.tipo": "cartesiano" | "polar" | "intrinseco" según convenga al problema;
  "rotacion" (solo cartesiano) = ángulo de los ejes (ej.: igual al del plano inclinado);
  "justificacion" = una frase de POR QUÉ conviene ese sistema.
- Si el sistema es "intrinseco": agregá "intrinseco": {"angulo_tangente": ...,
  "normal_hacia": "izquierda"|"derecha" (hacia el centro de curvatura), "radio": 1.3}.
- Si es "polar": agregá "polar": {"angulo_radial": ..., "distancia_origen": 1.5}
  (angulo_radial = dirección de r̂, de O hacia la partícula).
- "superficie": {"tipo": "horizontal"|"inclinada"|"techo"|"ninguna", "angulo": θ}.
- "cuerda": {"angulo": ..., "longitud": 1.3} SOLO si hay soga/cable/hilo visible.
- "cinematica": vectores v, a (y r si es polar) con "tipo": "velocidad"|"aceleracion"|"posicion".
  Si v = 0 o a = 0, no los incluyas.
- "notas": 1 a 3 frases cortas que ayuden a leer el diagrama (LaTeX permitido).

ESQUEMA EXACTO (completá los valores):
{
  "titulo": "DCL — <cuerpo y situación>",
  "cuerpo": "bloque" | "particula",
  "sistema": {"tipo": "cartesiano", "rotacion": 0, "justificacion": "..."},
  "superficie": {"tipo": "horizontal", "angulo": 0},
  "fuerzas": [
    {"nombre": "\\\\vec{N}", "angulo": 90, "magnitud": 1.0, "descomponer": false}
  ],
  "cinematica": [
    {"nombre": "\\\\vec{v}", "tipo": "velocidad", "angulo": 0, "magnitud": 1.0}
  ],
  "notas": ["..."]
}

EJEMPLO (bloque que baja deslizando por un plano inclinado de 30° con fricción):
{
  "titulo": "DCL — bloque sobre plano inclinado (θ = 30°)",
  "cuerpo": "bloque",
  "sistema": {"tipo": "cartesiano", "rotacion": 30,
              "justificacion": "ejes rotados: x' a lo largo del plano (dirección del movimiento) e y' perpendicular, así N y la aceleración quedan sobre los ejes"},
  "superficie": {"tipo": "inclinada", "angulo": 30},
  "fuerzas": [
    {"nombre": "\\\\vec{N}", "angulo": 120, "magnitud": 0.87},
    {"nombre": "\\\\vec{P}=m\\\\vec{g}", "angulo": -90, "magnitud": 1.0, "descomponer": true,
     "etiquetas_componentes": ["-mg\\\\sin\\\\theta", "-mg\\\\cos\\\\theta"]},
    {"nombre": "\\\\vec{f}_r", "angulo": 30, "magnitud": 0.3}
  ],
  "cinematica": [
    {"nombre": "\\\\vec{v}", "tipo": "velocidad", "angulo": 210, "magnitud": 1.0},
    {"nombre": "\\\\vec{a}", "tipo": "aceleracion", "angulo": 210, "magnitud": 0.5}
  ],
  "notas": ["El peso se descompone en los ejes rotados: mg\\\\sin\\\\theta y mg\\\\cos\\\\theta.",
            "La fricción cinética se opone al movimiento relativo: apunta plano arriba."]
}
""".strip()

_RE_JSON = re.compile(r'\{.*\}', re.DOTALL)


def _parsear_json(texto):
    """Extrae y parsea el primer objeto JSON de la respuesta del LLM."""
    if not texto:
        raise ValueError('respuesta vacía')
    t = texto.strip()
    t = re.sub(r'^```(?:json)?\s*|\s*```$', '', t, flags=re.MULTILINE).strip()
    m = _RE_JSON.search(t)
    if not m:
        raise ValueError('no se encontró un objeto JSON')
    return json.loads(m.group(0))


def _caption_dcl(spec):
    """Texto que acompaña al diagrama en el chat."""
    partes = []
    just = (spec.get('sistema', {}) or {}).get('justificacion', '')
    tipo = (spec.get('sistema', {}) or {}).get('tipo', 'cartesiano')
    if just:
        partes.append(f"**Sistema de coordenadas ({tipo}):** {just}")
    nombres = [f"${str(f.get('nombre','')).strip()}$" for f in spec.get('fuerzas', [])]
    if nombres:
        partes.append('**Fuerzas dibujadas:** ' + ', '.join(nombres))
    for n in (spec.get('notas') or [])[:3]:
        partes.append(f'• {n}')
    return '\n\n'.join(partes)


def generar_dcl(pregunta, respuesta=None, chunks=None, reintentos=1):
    """
    Genera el DCL para `pregunta`. Devuelve (ruta_png, spec, caption).
    Si ya tenés la respuesta del tutor, pasala en `respuesta` para que el
    diagrama sea coherente con la resolución.
    """
    if chunks is None:
        chunks = recuperar_contexto(pregunta, k=4, max_k=6)
    user = f"CONTEXTO:\n{_bloque_contexto(chunks)}\n\nPREGUNTA: {pregunta}"
    if respuesta:
        user += f"\n\nRESOLUCIÓN DEL TUTOR (para ser coherente):\n{respuesta[:2500]}"
    user += '\n\nJSON:'

    crudo = _llm([{'role': 'system', 'content': PROMPT_DCL},
                  {'role': 'user', 'content': user}],
                 temperature=0.1, max_tokens=1200)
    for intento in range(reintentos + 1):
        try:
            spec = _parsear_json(crudo)
            break
        except Exception as e:
            if intento >= reintentos:
                raise ValueError(f'JSON inválido del LLM: {e}')
            crudo = _llm([{'role': 'system', 'content': PROMPT_DCL},
                          {'role': 'user', 'content':
                           f"Este JSON es inválido ({e}). Corregilo y devolvé SOLO el JSON:\n{crudo}"}],
                         temperature=0.0, max_tokens=1200)

    ruta = dibujar_dcl(spec, mostrar=False)
    return ruta, spec, _caption_dcl(spec)


print('✅ generar_dcl() lista — ej.: ruta, spec, cap = generar_dcl("DCL de la caja del ejercicio 3")')

## C · Gráficas $x(t)$, $v(t)$, $a(t)$

Objetivo de la consigna para *asistentes avanzados*. Mismo patrón que el DCL: el LLM emite
un JSON con las **expresiones por tramo** (una fase del movimiento = un tramo), `sympy`
las parsea de forma segura y `matplotlib` grafica los tres paneles apilados con las
fronteras entre fases marcadas. En problemas multifase (como el ascensor de P5) la gráfica
*muestra* por qué el máximo no está al final de la fase 1.

In [ ]:
# ============================================================
#  GRÁFICAS x(t), v(t), a(t) — v1
#  El LLM emite un JSON con expresiones por tramo; acá se parsean
#  con sympy (seguro) y se grafican con matplotlib.
# ============================================================
import json, os, time
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

_T = sp.Symbol('t', real=True)
_LOCALS = {'t': _T, 'pi': sp.pi, 'sqrt': sp.sqrt, 'sin': sp.sin, 'cos': sp.cos,
           'tan': sp.tan, 'exp': sp.exp, 'log': sp.log, 'Abs': sp.Abs, 'abs': sp.Abs}

_COLORES = {'x': '#00838f', 'v': '#1565c0', 'a': '#2e7d32'}
_TITULOS = {'x': 'Posición  $x(t)$', 'v': 'Velocidad  $v(t)$', 'a': 'Aceleración  $a(t)$'}


def _fn(expr_str):
    """Parsea la expresión con sympy y devuelve una función numpy segura."""
    expr = sp.sympify(str(expr_str), locals=_LOCALS)
    f = sp.lambdify(_T, expr, modules='numpy')
    return lambda tt: np.broadcast_to(np.asarray(f(tt), dtype=float), np.shape(tt)).copy()


def graficar_cinematica(spec, ruta_png=None, mostrar=True):
    """
    spec = {
      "titulo": "...",
      "unidades": {"x": "m", "v": "m/s", "a": "m/s²", "t": "s"},
      "tramos": [
        {"t_ini": 0, "t_fin": 4, "x": "0.5*2*t**2", "v": "2*t", "a": "2",
         "etiqueta": "fase 1: MRUV"},
        ...
      ]
    }
    Las expresiones usan la variable t en SEGUNDOS y unidades del SI.
    Devuelve la ruta del PNG.
    """
    if isinstance(spec, str):
        spec = json.loads(spec)
    tramos = spec.get('tramos', [])
    if not tramos:
        raise ValueError('spec sin tramos')
    uni = spec.get('unidades', {}) or {}

    fig, axs = plt.subplots(3, 1, figsize=(7.6, 8.6), sharex=True)
    bordes = []
    for tr in tramos:
        t0, t1 = float(tr.get('t_ini', 0)), float(tr.get('t_fin', 1))
        if t1 <= t0:
            continue
        bordes.append(t1)
        tt = np.linspace(t0, t1, 250)
        for ax, var in zip(axs, ('x', 'v', 'a')):
            if tr.get(var) is None:
                continue
            try:
                yy = _fn(tr[var])(tt)
            except Exception as e:
                ax.text(0.02, 0.9, f'⚠ {var}: {e}', transform=ax.transAxes,
                        fontsize=8, color='crimson')
                continue
            ax.plot(tt, yy, color=_COLORES[var], lw=2.4)
        if tr.get('etiqueta'):
            axs[0].annotate(str(tr['etiqueta']), xy=((t0 + t1) / 2, 0),
                            xycoords=('data', 'axes fraction'),
                            xytext=(0, 6), textcoords='offset points',
                            ha='center', fontsize=9, color='#546e7a')

    for ax, var in zip(axs, ('x', 'v', 'a')):
        u = uni.get(var, {'x': 'm', 'v': 'm/s', 'a': 'm/s²'}[var])
        ax.set_ylabel(f'{var} [{u}]', fontsize=11)
        ax.set_title(_TITULOS[var], fontsize=11.5, loc='left', color=_COLORES[var])
        ax.grid(alpha=0.3)
        ax.axhline(0, color='#90a4ae', lw=0.9)
        for b in bordes[:-1]:
            ax.axvline(b, color='#b0bec5', lw=1, ls='--')
    axs[-1].set_xlabel(f"t [{uni.get('t', 's')}]", fontsize=11)

    fig.suptitle(spec.get('titulo', 'Cinemática del movimiento'), fontsize=13.5)
    fig.tight_layout(rect=[0, 0, 1, 0.97])

    if ruta_png is None:
        os.makedirs('figs', exist_ok=True)
        ruta_png = f'figs/cin_{int(time.time()*1000)}.png'
    fig.savefig(ruta_png, dpi=130, bbox_inches='tight', facecolor='white')
    if mostrar:
        plt.show()
    plt.close(fig)
    return ruta_png

In [ ]:
# ============================================================
#  GRÁFICAS x(t), v(t), a(t) — el LLM emite las expresiones,
#  sympy/numpy las evalúan y matplotlib grafica.
# ============================================================
PROMPT_GRAFICAS = """
Sos un experto en Cinemática. A partir del CONTEXTO, la PREGUNTA y (si está) la
RESOLUCIÓN, escribí las funciones x(t), v(t) y a(t) del movimiento como UN OBJETO
JSON. No escribas nada fuera del JSON.

REGLAS
- Variable temporal: t (en segundos). Unidades del SI. Usá g = 9.8 si hace falta,
  ya reemplazado numéricamente.
- Expresiones en sintaxis Python: potencias con **, multiplicación explícita con *.
  Podés usar sqrt, sin, cos, tan, exp, log, pi.
- TODOS los coeficientes numéricos ya evaluados (nada de símbolos sin valor).
- Un tramo por fase del movimiento, con t_ini y t_fin en segundos (t global,
  continuo entre tramos; en cada tramo la expresión usa el t GLOBAL, ej.: (t-4)).
- x(t) debe ser CONTINUA entre tramos (empalmá las constantes).
- "etiqueta": nombre corto de la fase.

ESQUEMA EXACTO:
{
  "titulo": "x(t), v(t) y a(t) — <situación>",
  "unidades": {"x": "m", "v": "m/s", "a": "m/s²", "t": "s"},
  "tramos": [
    {"t_ini": 0, "t_fin": 4, "x": "0.5*2*t**2", "v": "2*t", "a": "2",
     "etiqueta": "fase 1: MRUV"},
    {"t_ini": 4, "t_fin": 6.5, "x": "16 + 8*(t-4) - 4.9*(t-4)**2",
     "v": "8 - 9.8*(t-4)", "a": "-9.8", "etiqueta": "fase 2: caída libre"}
  ]
}
""".strip()


def generar_graficas(pregunta, respuesta=None, chunks=None, reintentos=1):
    """Genera las gráficas x(t), v(t), a(t). Devuelve (ruta_png, spec)."""
    if chunks is None:
        chunks = recuperar_contexto(pregunta, k=4, max_k=6)
    user = f"CONTEXTO:\n{_bloque_contexto(chunks)}\n\nPREGUNTA: {pregunta}"
    if respuesta:
        user += f"\n\nRESOLUCIÓN DEL TUTOR (usá ESTOS valores):\n{respuesta[:2500]}"
    user += '\n\nJSON:'

    crudo = _llm([{'role': 'system', 'content': PROMPT_GRAFICAS},
                  {'role': 'user', 'content': user}],
                 temperature=0.1, max_tokens=900)
    for intento in range(reintentos + 1):
        try:
            spec = _parsear_json(crudo)
            ruta = graficar_cinematica(spec, mostrar=False)
            return ruta, spec
        except Exception as e:
            if intento >= reintentos:
                raise ValueError(f'No pude construir las gráficas: {e}')
            crudo = _llm([{'role': 'system', 'content': PROMPT_GRAFICAS},
                          {'role': 'user', 'content':
                           f"Este JSON falló ({e}). Corregilo y devolvé SOLO el JSON:\n{crudo}"}],
                         temperature=0.0, max_tokens=900)


print('✅ generar_graficas() lista — ej.: ruta, spec = generar_graficas("graficá x(t) del ejercicio 15")')

## D · Interfaz Gradio v2

Reemplaza la celda 14 del v7. Novedades:

* Chat en formato `messages` con **imágenes en línea** (DCL y gráficas aparecen como
  mensajes del asistente, con su explicación de los ejes elegidos).
* El pedido de DCL / gráficas se detecta **automáticamente** en la pregunta
  ("hacé el DCL", "graficá x(t)"...) o se fuerza con los **checkboxes**.
* Checkbox "🔍 Mostrar verificación" → muestra el bloque Python y su salida (ideal para
  la presentación: se ve *en vivo* que los números están verificados).

> Si su versión de Gradio se queja del formato `messages`, cambien `type="messages"` por
> `type="tuples"` y las líneas `historial.append({"role": ..., "content": ...})` por el
> formato de tuplas `historial.append((pregunta, respuesta))` /
> `historial.append((None, (ruta,)))`.

In [ ]:
# ============================================================
#  INTERFAZ GRADIO v2 — chat + DCL + gráficas en línea
#  Reemplaza a la celda de Gradio de la versión 7.
# ============================================================
import gradio as gr

_RE_PIDE_DCL = re.compile(
    r'\b(dcl|diagrama\s+de\s+cuerpo\s+libre|cuerpo\s+libre|diagrama\s+de\s+fuerzas)\b',
    re.IGNORECASE)
_RE_PIDE_GRAF = re.compile(
    r'grafic\w*.{0,60}(posici[oó]n|velocidad|aceleraci[oó]n|x\s*\(\s*t|v\s*\(\s*t|a\s*\(\s*t)',
    re.IGNORECASE | re.DOTALL)


def chat_fisica_v2(pregunta, historial, con_dcl, con_graficas, mostrar_codigo):
    historial = list(historial or [])
    if not pregunta.strip():
        return historial, ""
    historial.append({"role": "user", "content": pregunta})

    respuesta = ask_physics_v2(pregunta, mostrar_codigo=mostrar_codigo)
    historial.append({"role": "assistant", "content": respuesta})

    if con_dcl or _RE_PIDE_DCL.search(pregunta):
        try:
            ruta, spec, caption = generar_dcl(pregunta, respuesta=respuesta)
            historial.append({"role": "assistant", "content": {"path": ruta}})
            if caption:
                historial.append({"role": "assistant", "content": caption})
        except Exception as e:
            historial.append({"role": "assistant",
                              "content": f"⚠ No pude generar el DCL: {e}"})

    if con_graficas or _RE_PIDE_GRAF.search(pregunta):
        try:
            ruta, _ = generar_graficas(pregunta, respuesta=respuesta)
            historial.append({"role": "assistant", "content": {"path": ruta}})
        except Exception as e:
            historial.append({"role": "assistant",
                              "content": f"⚠ No pude generar las gráficas: {e}"})

    return historial, ""


with gr.Blocks(title="Asistente de Física — Mecánica: FIA") as demo:
    gr.Markdown(
        "# 🧪 FIA: Asistente de Física — Mecánica\n"
        "Preguntá por teoría o ejercicios de la guía. Los cálculos numéricos se "
        "**verifican con Python** y podés pedir el **DCL** o las **gráficas x(t), v(t), a(t)**."
    )

    chatbot = gr.Chatbot(
        label="Conversación", height=520,
        latex_delimiters=[
            {"left": "$$", "right": "$$", "display": True},
            {"left": "$",  "right": "$",  "display": False},
            {"left": "\\[", "right": "\\]", "display": True},
            {"left": "\\(", "right": "\\)", "display": False},
        ],
    )

    with gr.Row():
        entrada = gr.Textbox(
            placeholder="Ej: Resolvé el ejercicio 3 de la guía 2 y hacé el DCL...",
            label="Tu pregunta", scale=4)
        boton = gr.Button("Preguntar", variant="primary", scale=1)

    with gr.Row():
        chk_dcl = gr.Checkbox(label="📐 Incluir DCL", value=False)
        chk_graf = gr.Checkbox(label="📈 Incluir gráficas x(t), v(t), a(t)", value=False)
        chk_cod = gr.Checkbox(label="🔍 Mostrar verificación (Python)", value=False)

    gr.Examples(
        examples=[
            "Un camión frena desde 70 km/h con una caja sobre la plataforma (μs = 0,3). Hacé el DCL y hallá la distancia mínima de frenado sin que la caja resbale.",
            "Resolvé el ejercicio 12 de la guía y hacé el diagrama de cuerpo libre",
            "¿Por qué en el MCU hay aceleración si el módulo de la velocidad es constante?",
            "Un bloque baja por un plano inclinado de 30° con fricción: DCL con los ejes que convenga usar",
            "Graficá x(t), v(t) y a(t) de un ascensor que sube con a = 2 m/s² durante 4 s y luego se corta el cable",
        ],
        inputs=entrada,
    )

    boton.click(chat_fisica_v2,
                inputs=[entrada, chatbot, chk_dcl, chk_graf, chk_cod],
                outputs=[chatbot, entrada])
    entrada.submit(chat_fisica_v2,
                   inputs=[entrada, chatbot, chk_dcl, chk_graf, chk_cod],
                   outputs=[chatbot, entrada])

# share=True genera URL pública válida 72 hs
demo.launch(share=True, debug=False)

## Notas para el informe y la presentación

**Qué contar de las mejoras (mapea 1-a-1 con los comentarios del profesor):**

1. *"Fragmentos"* → explicar en el informe: un *fragmento (chunk)* es un pedacito de texto
   (≈ un párrafo) de los libros/guías; el asistente busca los más parecidos a la pregunta
   y se los pasa al modelo como contexto.
2. *Mecánica en lugar de cinemática* → el benchmark E ya usa problemas de dinámica
   (fricción, plano inclinado, péndulo) y el system prompt cubre todo el alcance de la
   consigna (dinámica, trabajo-energía, p, L).
3. *¿Se logró lo planificado?* → estructura sugerida: (a) verificación numérica: lograda,
   mostrar P5 antes/después; (b) DCL: logrado, mostrar 2 sistemas de ejes distintos;
   (c) gráficas x/v/a: logradas; (d) lo que quede pendiente (p. ej. filtrar chunks OCR
   ruidosos) decir por qué y cómo seguiría.

**Para la demo en vivo:** pedir el ejercicio de la caja del camión con el checkbox de DCL
y el de verificación activados — en una sola pregunta se ven las dos mejoras.

**Limitaciones honestas para mencionar:** el DCL depende de que el LLM identifique bien
las fuerzas (el renderizador dibuja lo que el JSON dice); si el bloque CALC falla dos
veces, la respuesta sale con una advertencia visible en lugar de números inventados;
la detección de "pedime un DCL" es por palabras clave + checkbox (no semántica).